# 07. 사람 개입(HITL)과 런타임

## 학습 목표

도구 실행 전 사람의 승인을 받고, 런타임 컨텍스트를 주입합니다.

이 노트북에서 다루는 내용:
- **Human-in-the-Loop (HITL)**: 에이전트가 위험한 도구를 실행하기 전에 사람의 승인을 받는 패턴
- **ToolRuntime**: 도구 실행 시 런타임 컨텍스트(사용자 정보 등)를 주입하는 방법
- **컨텍스트 엔지니어링**: 동적으로 프롬프트와 도구를 제어하는 기법
- **MCP (Model Context Protocol)**: 도구 서버를 표준 프로토콜로 연결하는 방식

## 7.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

모델 준비 완료: gpt-4.1


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 7.2 Human-in-the-Loop 개념

에이전트가 도구를 호출하기 전에 사람의 승인을 요청합니다.

### 왜 필요한가?

자율적으로 동작하는 에이전트는 강력하지만, 이메일 전송, 파일 삭제, 결제 처리 같은 **되돌릴 수 없는 작업**에서는 사람의 확인이 필수적입니다.

### 워크플로

```
에이전트 → 도구 호출 제안 → [중단(interrupt)] → 사람 승인/거부 → 도구 실행 → 결과 반환
```

LangChain v1에서는 `HumanInTheLoopMiddleware`와 `InMemorySaver`(체크포인터)를 결합하여 이 패턴을 구현합니다. 체크포인터는 에이전트의 상태를 저장하여 중단 후 재개할 수 있게 합니다.

## 7.3 HumanInTheLoopMiddleware

`HumanInTheLoopMiddleware`는 도구 호출 시 자동으로 실행을 중단하고, 사람의 승인을 기다리는 미들웨어입니다. `InMemorySaver` 체크포인터와 함께 사용하여 중단된 상태를 보존합니다.

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """지정된 수신자에게 이메일을 보냅니다."""
    return f"{to}에게 이메일 전송 완료: {subject}"

@tool
def delete_file(path: str) -> str:
    """지정된 경로의 파일을 삭제합니다."""
    return f"파일 삭제 완료: {path}"

# 위험한 도구에만 승인 요구
hitl = HumanInTheLoopMiddleware(interrupt_on={
    "send_email": True,
    "delete_file": True,
})

agent = create_agent(
    model=model,
    tools=[send_email, delete_file],
    system_prompt="당신은 이메일을 보내고 파일을 관리할 수 있는 어시스턴트입니다.",
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("HITL 에이전트 생성 완료")
print("  -> 도구 호출 시 사람의 승인을 위해 중단됩니다")

HITL 에이전트 생성 완료
  -> 도구 호출 시 사람의 승인을 위해 중단됩니다


## 7.4 interrupt와 Command(resume=...) 패턴

HITL 에이전트는 2단계로 동작합니다:

1. **1단계 (invoke)**: 에이전트가 도구 호출을 제안하면 자동으로 **중단(interrupt)**됩니다.
2. **2단계 (Command(resume=True))**: 사람이 승인하면 `Command(resume=True)`로 실행을 **재개**합니다.

거부할 경우 `Command(resume=False)`를 사용하거나, 다른 지시를 줄 수 있습니다.

In [4]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "hitl-demo"}}

# 1단계: 에이전트 실행 -> 도구 호출에서 중단
result = agent.invoke(
    {"messages": [{"role": "user", "content": "bob@example.com에게 제목 '인사', 본문 '안녕하세요 Bob!' 이메일을 보내주세요"}]},
    config={**config, **lf_config},
)

# 중단 상태 확인
print("현재 상태:", result)
print("\n도구 실행 전에 에이전트가 중단되었습니다.")
print("승인하려면 Command(resume=True)를 사용하세요.")

# 2단계: 승인하여 계속 진행
try:
    result = agent.invoke(Command(resume=True), config={**config, **lf_config})
    print("\n승인 후 결과:", result["messages"][-1].content)
except Exception as e:
    print(f"\n참고: HITL 데모는 인터랙티브 환경에서 가장 잘 동작합니다. ({e})")

현재 상태: {'messages': [HumanMessage(content="bob@example.com에게 제목 '인사', 본문 '안녕하세요 Bob!' 이메일을 보내주세요", additional_kwargs={}, response_metadata={}, id='cccd35e6-d0bc-45c3-a7ff-65ac013e67ab'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 121, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_e7424371d3', 'id': 'chatcmpl-DGQzhxBh0PhTmumZMfGCHbvQTX9UV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cc3ac-75de-7aa0-b9a6-9c9618a00daf-0', tool_calls=[{'name': 'send_email', 'args': {'to': 'bob@example.com', 'subject': '인사', 'body': '안녕하세요 Bob!'}, 'id': 'call_mMkjJDZJTQILGRDjR0dflEHk', 'type': 'tool_ca

## 7.5 ToolRuntime -- 도구에서 런타임 정보에 접근합니다

`ToolRuntime`은 도구가 실행될 때 런타임 컨텍스트(현재 사용자 정보, 세션 데이터 등)에 접근할 수 있게 해주는 메커니즘입니다.

### 핵심 아이디어
- 도구 함수에 `runtime: ToolRuntime[T]` 파라미터를 추가합니다.
- `T`는 개발자가 정의하는 컨텍스트 데이터 클래스입니다.
- 에이전트 생성 시 `context_schema=T`를 지정하고, 호출 시 `context=T(...)`로 값을 전달합니다.

In [5]:
from langchain.tools import ToolRuntime
from dataclasses import dataclass

@dataclass
class UserContext:
    """사용자 정보가 포함된 런타임 컨텍스트."""
    user_id: str
    role: str

@tool
def get_user_profile(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 프로필 정보를 가져옵니다."""
    ctx = runtime.context
    return f"사용자 ID: {ctx.user_id}, 역할: {ctx.role}"

@tool
def check_permissions(action: str, runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자가 작업에 대한 권한이 있는지 확인합니다."""
    ctx = runtime.context
    if ctx.role == "admin":
        return f"사용자 {ctx.user_id}은(는) '{action}' 권한이 있습니다"
    return f"사용자 {ctx.user_id}은(는) '{action}' 권한이 없습니다"

agent_ctx = create_agent(
    model=model,
    tools=[get_user_profile, check_permissions],
    system_prompt="사용자 프로필과 권한을 확인할 수 있습니다.",
    context_schema=UserContext,
)

result = agent_ctx.invoke(
    {"messages": [{"role": "user", "content": "제가 누구이고 파일을 삭제할 수 있나요?"}]},
    context=UserContext(user_id="user-42", role="admin"),
    config=lf_config,
)
print("결과:", result["messages"][-1].content)

결과: 당신은 사용자 ID가 user-42이고, 역할은 admin입니다.
또한, 파일 삭제(delete_file) 권한이 있습니다. 따라서 파일을 삭제할 수 있습니다.


## 7.6 컨텍스트 엔지니어링 -- 동적으로 프롬프트와 도구를 제어합니다

컨텍스트 엔지니어링은 에이전트에게 전달되는 **프롬프트**, **도구**, **메시지 히스토리**를 동적으로 조작하는 기법입니다.

### 주요 활용 사례
- 사용자 역할에 따라 다른 시스템 프롬프트 제공
- 상황에 따라 사용 가능한 도구 필터링
- 긴 대화 히스토리 요약 및 정리

`dynamic_prompt` 미들웨어를 사용하면 매 요청마다 프롬프트를 커스터마이즈할 수 있습니다.

In [6]:
from langchain.agents.middleware import dynamic_prompt

@tool
def basic_search(query: str) -> str:
    """기본 웹 검색을 수행합니다."""
    return f"'{query}'에 대한 기본 검색 결과"

@tool
def advanced_analytics(query: str) -> str:
    """고급 데이터 분석을 수행합니다."""
    return f"'{query}'에 대한 분석 보고서"

# 사용자 역할에 따라 다른 프롬프트와 도구 제공
@dynamic_prompt
def role_based_prompt(request):
    """컨텍스트에 기반하여 프롬프트를 커스터마이즈합니다."""
    return "당신은 전문 어시스턴트입니다. 사용자의 질문에 효율적으로 답변하세요."

agent_ctx_eng = create_agent(
    model=model,
    tools=[basic_search, advanced_analytics],
    middleware=[role_based_prompt],
)

result = agent_ctx_eng.invoke(
    {"messages": [{"role": "user", "content": "머신러닝 트렌드를 검색해 주세요"}]},
    config=lf_config,
)
print("컨텍스트 엔지니어링 결과:", result["messages"][-1].content[:200])

컨텍스트 엔지니어링 결과: 최신 머신러닝 트렌드에 대해 검색한 결과를 요약해 드리겠습니다:

1. 생성형 AI(Generative AI)의 급부상:
   - 챗GPT 등 대형 언어 모델(LLM)을 활용한 텍스트, 이미지, 음악 등 다양한 콘텐츠 생성 기술이 주목받고 있습니다.

2. 멀티모달 AI:
   - 텍스트, 음성, 이미지 등 여러 형태의 데이터를 동시에 처리하고 이해하는 멀


## 7.7 MCP (Model Context Protocol) 연동 개요

**MCP**는 도구 서버를 표준 프로토콜로 연결하는 방식입니다.

### MCP의 핵심 개념
- **MCP 서버**: 도구(Tool)를 제공하는 서버. HTTP/SSE 또는 stdio로 통신합니다.
- **MCP 클라이언트**: 에이전트가 MCP 서버에 연결하여 도구를 발견하고 호출합니다.
- **표준화**: 어떤 언어/프레임워크로 만든 도구든 MCP 프로토콜을 따르면 연결 가능합니다.

### LangChain v1에서의 MCP 지원
- `mcp.client.stdio.stdio_client()`와 `ClientSession`으로 로컬 MCP 서버에 연결할 수 있습니다.
- `langchain-mcp-adapters`의 `load_mcp_tools(session)`로 MCP 세션의 도구를 LangChain Tool로 변환할 수 있습니다.

In [ ]:
from pathlib import Path; import asyncio, tempfile, sys
from mcp import ClientSession, StdioServerParameters; from mcp.client.stdio import stdio_client; from langchain_mcp_adapters.tools import load_mcp_tools
server_path = Path(tempfile.gettempdir()) / "lc_mcp_echo_server.py"
server_path.write_text('from mcp.server.fastmcp import FastMCP\nmcp = FastMCP("echo")\n@mcp.tool()\ndef echo(text: str) -> str:\n    return f"Echo: {text}"\nif __name__ == "__main__":\n    mcp.run(transport="stdio")')
async def run_mcp_agent():
    params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
    async with stdio_client(params) as (read, write), ClientSession(read, write) as session: await session.initialize(); tools = await load_mcp_tools(session); agent = create_agent(model=model, tools=tools, system_prompt="MCP 도구를 사용할 수 있습니다."); return await agent.ainvoke({"messages": [{"role": "user", "content": "echo 도구로 '안녕하세요'를 반복해 주세요."}]}, config=lf_config)
result = asyncio.run(run_mcp_agent()); print(result["messages"][-1].content)

## 7.8 요약

이 노트북에서 학습한 핵심 내용:

| 개념 | 설명 | 핵심 API |
|------|------|----------|
| **HITL** | 도구 실행 전 사람의 승인 요청 | `HumanInTheLoopMiddleware`, `Command(resume=...)` |
| **ToolRuntime** | 도구에서 런타임 컨텍스트 접근 | `ToolRuntime[T]`, `context_schema` |
| **컨텍스트 엔지니어링** | 동적 프롬프트/도구 제어 | `dynamic_prompt` 미들웨어 |
| **MCP** | 표준화된 도구 프로토콜 | `ClientSession + load_mcp_tools()` |

### 다음 단계
- 다음 노트북에서는 **멀티 에이전트 패턴**을 학습합니다.
- 서브에이전트, 핸드오프, 스킬, 라우터 등 다양한 에이전트 협업 방식을 다룹니다.